# PatchTST — Walmart Store Sales Forecasting

MLflow experiment: `PatchTST_Training`.
Runs: `PatchTST_DataPrep`, `PatchTST_CrossValidation`, `PatchTST_HPO`, `PatchTST_Final`.

Global transformer over patched series via Nixtla's `neuralforecast`, trained with the
exogenous features from the shared `features.build_features`. Metric: WMAE, holiday weeks
weighted 5x. Validation: expanding window, horizon = 39 weeks (same folds as the tree models).

There is no separate `nf_prep.py` in this repo yet (see `TASKS.md` T13), so the Nixtla
long-format conversion is done inline in this notebook (`to_nixtla` / `from_nixtla` below).
If `nf_prep.py` lands later, swap these two functions for the shared import.

## Environment

In [ ]:
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install mlflow dagshub kaggle neuralforecast optuna pandas pyarrow python-dotenv
    import os
    if not os.path.isdir("ML-final"):
        !git clone https://github.com/ketevan614/ML-final.git
    %cd ML-final

    from google.colab import userdata
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        f.write(userdata.get("KAGGLE_JSON"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    if not os.path.exists("data/train.csv"):
        !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
        import zipfile, glob
        os.chdir("data")
        while glob.glob("*.zip"):
            for z in glob.glob("*.zip"):
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(".")
                os.remove(z)
        os.chdir("..")
print("IN_COLAB =", IN_COLAB)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import os
    if not os.path.isdir("ML-final"):
        !git clone https://github.com/ketevan614/ML-final.git
    %cd ML-final

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
print("IN_COLAB =", IN_COLAB, "| ROOT =", ROOT)


## Robust data directory (Fix 1)

In [ ]:
from pathlib import Path

def find_data_dir(root):
    """Locate the folder that actually holds the 5 Kaggle CSVs.

    Handles both layouts: a flat data/ folder, or the nested folder Kaggle's
    zip produces (data/walmart-recruiting-store-sales-forecasting/).
    """
    required = {"train.csv", "test.csv", "features.csv", "stores.csv", "sampleSubmission.csv"}
    for d in [root / "data", root / "data" / "walmart-recruiting-store-sales-forecasting"]:
        if d.exists() and required.issubset({p.name for p in d.glob("*.csv")}):
            return d
    raise FileNotFoundError("Could not find all 5 CSVs in data/ or the nested Kaggle folder.")

DATA_DIR = find_data_dir(ROOT)
print("DATA_DIR =", DATA_DIR)


## MLflow / DagsHub tracking (Fix 2)

In [ ]:
import os
import mlflow

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    print("python-dotenv not installed; relying on already-exported env vars")

# stale proxy vars have previously broken the DagsHub connection in Colab
for var in ("HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"):
    os.environ.pop(var, None)

EXPERIMENT_NAME = "PatchTST_Training"

def setup_tracking():
    """Use DagsHub MLflow if creds are present in the env / .env, else fall
    back to a local file-based tracking store so a bad token or a network
    hiccup never blocks a training run."""
    uri = os.environ.get("MLFLOW_TRACKING_URI")
    if uri and os.environ.get("MLFLOW_TRACKING_PASSWORD"):
        try:
            mlflow.set_tracking_uri(uri)
            mlflow.set_experiment(EXPERIMENT_NAME)
            print("MLflow tracking ->", uri)
            return
        except Exception as e:
            print("DagsHub tracking unavailable, falling back to local mlruns:", e)
    fallback_uri = f"file:{(ROOT / 'mlruns').as_posix()}"
    mlflow.set_tracking_uri(fallback_uri)
    mlflow.set_experiment(EXPERIMENT_NAME)
    print("MLflow tracking -> local", fallback_uri)

setup_tracking()

def safe_log_params(params):
    try:
        mlflow.log_params(params)
    except Exception as e:
        print("mlflow.log_params failed:", e)

def safe_log_param(key, value):
    try:
        mlflow.log_param(key, value)
    except Exception as e:
        print(f"mlflow.log_param({key}) failed:", e)

def safe_log_metric(key, value):
    try:
        mlflow.log_metric(key, value)
    except Exception as e:
        print(f"mlflow.log_metric({key}) failed:", e)

def safe_log_artifact(path):
    try:
        mlflow.log_artifact(path)
    except Exception as e:
        print(f"mlflow.log_artifact({path}) failed:", e)


## Imports and data

In [ ]:
import numpy as np
import pandas as pd

from dataloader import load_raw, merge_all, make_submission
from features import build_features
from metrics import wmae
from validation import expanding_window_folds

try:
    from neuralforecast import NeuralForecast
    from neuralforecast.models import PatchTST
    from neuralforecast.losses.pytorch import MAE
except ImportError as e:
    raise ImportError(
        "neuralforecast is required for this notebook. Install requirements-dl.txt "
        "in its own environment (see README) before running this cell."
    ) from e

train, test, features, stores, sample = load_raw(DATA_DIR)
train_merged = merge_all(train, features, stores).reset_index(drop=True)
raw_test = test[["Store", "Dept", "Date", "IsHoliday"]].reset_index(drop=True)
dates = train["Date"].to_numpy()
print("train_merged:", train_merged.shape, "| raw_test:", raw_test.shape)

## Run: PatchTST_DataPrep

Convert to Nixtla long format (`unique_id`, `ds`, `y` + numeric exogenous columns).
Series with fewer than `MIN_HISTORY` weeks are dropped from *training* only (too short to
learn from) but every `(Store, Dept)` still gets a forecast at inference time - see the
fallback in `from_nixtla`/the pyfunc wrapper.

In [ ]:
EXOG_COLS_CACHE = {}

def _numeric_exog(df_merged):
    """Build the shared feature matrix and keep only numeric, non-target columns
    (neuralforecast exogenous features must be numeric)."""
    X = build_features(df_merged, sales_history_df=None, encode_categoricals=True)
    X = X.select_dtypes(include=[np.number]).astype("float32")
    return X

def to_nixtla(df_merged, fit_columns=None):
    """Raw merged frame -> Nixtla long format. If `fit_columns` is given, reindex to it
    (so train/future frames share identical exogenous columns)."""
    X = _numeric_exog(df_merged)
    if fit_columns is not None:
        X = X.reindex(columns=fit_columns, fill_value=0)
    nf = pd.DataFrame({
        "unique_id": df_merged["Store"].astype(str) + "_" + df_merged["Dept"].astype(str),
        "ds": pd.to_datetime(df_merged["Date"]),
    })
    if "Weekly_Sales" in df_merged.columns:
        nf["y"] = df_merged["Weekly_Sales"].to_numpy(dtype=float)
    for col in X.columns:
        nf[col] = X[col].to_numpy()
    return nf, list(X.columns)

MIN_HISTORY = 52  # need at least one full year to be useful to a global model

nf_train_full, EXOG_COLS = to_nixtla(train_merged)
series_counts = nf_train_full.groupby("unique_id")["ds"].count()
short_series = series_counts[series_counts < MIN_HISTORY].index
nf_train = nf_train_full[~nf_train_full["unique_id"].isin(short_series)].reset_index(drop=True)

with mlflow.start_run(run_name="PatchTST_DataPrep"):
    safe_log_param("min_history_weeks", MIN_HISTORY)
    safe_log_param("n_exogenous_cols", len(EXOG_COLS))
    safe_log_param("n_series_total", series_counts.shape[0])
    safe_log_param("n_series_dropped_short", len(short_series))
    safe_log_metric("nf_train_rows", float(len(nf_train)))
    print(f"series: {series_counts.shape[0]} total, {len(short_series)} dropped (<{MIN_HISTORY}w)")
    print("exogenous columns:", len(EXOG_COLS))

## Run: PatchTST_CrossValidation

Global model trained on the expanding-window folds, comparing **with vs without exogenous**
features. `NeuralForecast.cross_validation` isn't used directly because our folds must match
the tree-model folds exactly (`validation.expanding_window_folds`); instead each fold is fit
and predicted explicitly.

In [ ]:
FOLD_DATE_RANGES = list(expanding_window_folds(dates, n_splits=3, horizon=39))

def run_patchtst_fold(tr_mask, va_mask, use_exog, model_kwargs):
    tr = train_merged[tr_mask].reset_index(drop=True)
    va = train_merged[va_mask].reset_index(drop=True)

    nf_tr, fit_cols = to_nixtla(tr)
    short = nf_tr.groupby("unique_id")["ds"].count()
    short = short[short < MIN_HISTORY].index
    nf_tr = nf_tr[~nf_tr["unique_id"].isin(short)].reset_index(drop=True)

    hist_cols = fit_cols if use_exog else []
    futr_cols = hist_cols

    model = PatchTST(
        h=39,
        loss=MAE(),
        futr_exog_list=futr_cols,
        hist_exog_list=hist_cols,
        **model_kwargs,
    )
    nf = NeuralForecast(models=[model], freq="W-FRI")
    cols_for_fit = ["unique_id", "ds", "y"] + hist_cols
    nf.fit(df=nf_tr[cols_for_fit])

    nf_va, _ = to_nixtla(va, fit_columns=fit_cols)
    futr_df = nf_va[["unique_id", "ds"] + futr_cols] if futr_cols else nf_va[["unique_id", "ds"]]
    preds = nf.predict(futr_df=futr_df).reset_index()

    merged = nf_va[["unique_id", "ds", "y"]].merge(
        preds[["unique_id", "ds", "PatchTST"]], on=["unique_id", "ds"], how="left"
    )
    fallback = float(nf_tr["y"].mean())
    merged["PatchTST"] = merged["PatchTST"].fillna(fallback)
    p = np.clip(merged["PatchTST"].to_numpy(), 0, None)

    va_key = va.assign(unique_id=va["Store"].astype(str) + "_" + va["Dept"].astype(str))
    holiday_lookup = va_key.set_index(["unique_id", "Date"])["IsHoliday"].astype(bool)
    hva = merged.set_index(["unique_id", "ds"]).index.map(
        lambda k: holiday_lookup.get(k, False)
    )
    return wmae(merged["y"].to_numpy(), p, np.asarray(hva, dtype=bool))

BASE_MODEL_KWARGS = dict(input_size=104, patch_len=16, stride=8, hidden_size=128,
                        n_heads=4, max_steps=200, val_check_steps=50, random_seed=42)

with mlflow.start_run(run_name="PatchTST_CrossValidation"):
    scores_exog, scores_noexog = [], []
    for tr_mask, va_mask in FOLD_DATE_RANGES:
        scores_exog.append(run_patchtst_fold(tr_mask, va_mask, True, BASE_MODEL_KWARGS))
        scores_noexog.append(run_patchtst_fold(tr_mask, va_mask, False, BASE_MODEL_KWARGS))
    mean_exog = float(np.mean(scores_exog))
    mean_noexog = float(np.mean(scores_noexog))
    safe_log_params({"input_size": BASE_MODEL_KWARGS["input_size"],
                     "patch_len": BASE_MODEL_KWARGS["patch_len"],
                     "stride": BASE_MODEL_KWARGS["stride"]})
    for i, (we, wo) in enumerate(zip(scores_exog, scores_noexog)):
        safe_log_metric(f"wmae_fold_{i}_exog", we)
        safe_log_metric(f"wmae_fold_{i}_noexog", wo)
    safe_log_metric("wmae_cv_mean_exog", mean_exog)
    safe_log_metric("wmae_cv_mean_noexog", mean_noexog)
    print(f"CV WMAE  exog={mean_exog:,.1f}   no-exog={mean_noexog:,.1f}")

USE_EXOG = mean_exog <= mean_noexog
print("USE_EXOG =", USE_EXOG)

## Run: PatchTST_HPO

Optuna search over the main PatchTST hyperparameters; each trial is logged as a nested run.
Uses only the **last** fold to keep trial cost reasonable (same trade-off the tree-model
notebooks make for feature selection).

In [ ]:
import optuna

N_TRIALS = 5
LAST_FOLD = FOLD_DATE_RANGES[-1]

def objective(trial):
    kwargs = dict(
        input_size=trial.suggest_categorical("input_size", [52, 78, 104, 156]),
        patch_len=trial.suggest_categorical("patch_len", [8, 16, 24]),
        stride=trial.suggest_categorical("stride", [4, 8, 12]),
        hidden_size=trial.suggest_categorical("hidden_size", [64, 128, 256]),
        n_heads=trial.suggest_categorical("n_heads", [2, 4, 8]),
        max_steps=200,
        val_check_steps=50,
        random_seed=42,
    )
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        score = run_patchtst_fold(LAST_FOLD[0], LAST_FOLD[1], USE_EXOG, kwargs)
        safe_log_params(kwargs)
        safe_log_metric("wmae_cv_mean", score)
    return score

with mlflow.start_run(run_name="PatchTST_HPO"):
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=N_TRIALS)
    safe_log_params(study.best_params)
    safe_log_metric("wmae_cv_best", study.best_value)
    print("best WMAE:", round(study.best_value, 1))
    print("best params:", study.best_params)

BEST_MODEL_KWARGS = dict(BASE_MODEL_KWARGS, **study.best_params, max_steps=300)

## Run: PatchTST_Final

Fit on the full training history and log a raw->prediction `pyfunc` (via
`neuralforecast_pyfunc.NeuralForecastRawPyFunc`), matching the Pipeline contract the tree
models use. This is what `model_inference.ipynb` loads.

In [ ]:
from neuralforecast_pyfunc import NeuralForecastRawPyFunc

nf_train_final, FINAL_FIT_COLS = to_nixtla(train_merged)
final_short = nf_train_final.groupby("unique_id")["ds"].count()
final_short = final_short[final_short < MIN_HISTORY].index
nf_train_final = nf_train_final[~nf_train_final["unique_id"].isin(final_short)].reset_index(drop=True)

hist_cols = FINAL_FIT_COLS if USE_EXOG else []
final_model = PatchTST(
    h=39, loss=MAE(), futr_exog_list=hist_cols, hist_exog_list=hist_cols,
    **{k: v for k, v in BEST_MODEL_KWARGS.items()},
)
nf_final = NeuralForecast(models=[final_model], freq="W-FRI")
nf_final.fit(df=nf_train_final[["unique_id", "ds", "y"] + hist_cols])

pyfunc_model = NeuralForecastRawPyFunc(
    neuralforecast_model=nf_final,
    features_df=features,
    stores_df=stores,
    train_history_raw=train,
    model_alias="PatchTST",
    uses_exog=USE_EXOG,
)

test_pred = pyfunc_model.predict(None, raw_test)
print("predicted test rows:", test_pred.shape, "| sample:", np.round(test_pred[:5], 1))

with mlflow.start_run(run_name="PatchTST_Final"):
    safe_log_params(BEST_MODEL_KWARGS)
    safe_log_param("use_exog", USE_EXOG)
    final_mean_wmae = float(np.mean(
        [run_patchtst_fold(tr, va, USE_EXOG, BEST_MODEL_KWARGS) for tr, va in FOLD_DATE_RANGES]
    ))
    safe_log_metric("wmae_cv_mean", final_mean_wmae)
    import mlflow.pyfunc
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=pyfunc_model,
        code_paths=["neuralforecast_pyfunc.py", "features.py", "dataloader.py"],
        input_example=raw_test.head(3),
    )
    print("logged pyfunc | CV WMAE:", round(final_mean_wmae, 1))

make_submission(raw_test, test_pred, "submission_patchtst.csv")
print("wrote submission_patchtst.csv")

## Results

- CV WMAE: ___  | Kaggle public LB WMAE: ___
- Exogenous features helped: ___
- Best HPO config: ___
- vs LightGBM / XGBoost: ___
- vs DLinear / N-BEATS: ___
